# IV Mean Reversion – Backtesting Notebook
Loads IV data, computes mean-reversion signals, and backtests a short-vol strategy.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent / 'src'))

import pandas as pd
import numpy as np
import scipy.stats as stats
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from iv_mr.iv_utils import iv_zscore, iv_percentile, half_life

sns.set_theme(style='darkgrid')
print('Imports OK')

## 1. Load Market Data (yfinance fallback)
Replace with `load_iv_snapshot()` once IB data is collected.

In [ ]:
SYMBOL = 'SPY'
raw = yf.download(SYMBOL, period='2y', auto_adjust=True)
# Proxy IV: 30-day realised vol (annualised) as placeholder until IB IV is available
log_ret = np.log(raw['Close'] / raw['Close'].shift(1))
rv30 = log_ret.rolling(21).std() * np.sqrt(252) * 100   # in %
rv30.name = 'IV_proxy'
rv30 = rv30.dropna()
rv30.tail()

## 2. Mean-Reversion Signals

In [ ]:
z = iv_zscore(rv30, window=20)
pct = iv_percentile(rv30, window=252)
hl = half_life(rv30)
print(f'Half-life of mean reversion: {hl:.1f} trading days')

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
rv30.plot(ax=axes[0], title='IV Proxy (30-day RV %)', color='steelblue')
z.plot(ax=axes[1], title='Rolling Z-Score (20d)', color='darkorange')
axes[1].axhline(1.5, ls='--', color='red', lw=0.8)
axes[1].axhline(-1.5, ls='--', color='green', lw=0.8)
pct.plot(ax=axes[2], title='IV Percentile (252d)', color='purple')
axes[2].axhline(80, ls='--', color='red', lw=0.8)
axes[2].axhline(20, ls='--', color='green', lw=0.8)
plt.tight_layout()
plt.show()

## 3. Simple Short-Vol Backtest
**Rule:** Short vol (sell premium) when IV z-score > 1.5; flat otherwise.  
PnL proxy: next-day change in RV (negative is profit for short-vol).

In [ ]:
signal = (z > 1.5).astype(int)  # 1 = short vol
daily_pnl = -signal.shift(1) * rv30.diff()  # short vol profits when IV falls
cum_pnl = daily_pnl.cumsum()

sharpe = daily_pnl.mean() / daily_pnl.std() * np.sqrt(252)
print(f'Annualised Sharpe: {sharpe:.2f}')

cum_pnl.plot(figsize=(14, 4), title='Cumulative PnL – Short IV Mean Reversion', color='teal')
plt.ylabel('Cumulative IV pts')
plt.tight_layout()
plt.show()